In [ ]:
import os
import json
import time
from typing import Dict, Any, Tuple, List, Optional

from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from tqdm import tqdm

from google import genai
from google.genai import types

api_key = "" 
client = genai.Client(api_key=api_key)

In [9]:
# 任务说明（可按需微调）
INSTRUCTION = (
    "Please determine whether there is a non-crash functional bug in the following app operation log, "
    "and explain the sequence by analyzing the meaning and result of each recorded operation.\n"
    "output format {\n\t\"is_bug\": true or false,\n\t\"reason\": brief explain why do you consider it is a bug or it does not has bug\n}"
    "\nUI info interaction trace:\n"
)

# 模型与生成配置
MODEL_NAME = "gemini-2.5-pro"
GEN_CONFIG = types.GenerateContentConfig(
    temperature=0.3,
    max_output_tokens=2048,
    thinking_config=types.ThinkingConfig(include_thoughts=True)  # 关键
)

# 控制参数
RATE_PAUSE_SEC  = 2           # 每次成功调用后的节流
MAX_ELEMENTS    = 10000        # 如需限制最大处理条数，可改为具体数字
EXCLUDED_IDS    = set()        # 需要跳过的 id（可自定义）
FAIL_SLEEP_SEC  = 10           # 失败后的休眠秒数
MAX_CONSEC_FAIL = 10           # 连续失败上限


In [10]:
def extract_thoughts_and_answer(response) -> Tuple[List[str], str]:
    thoughts: List[str] = []
    answers: List[str] = []

    try:
        cand = response.candidates[0]
        parts = getattr(cand.content, "parts", []) or []

        for part in parts:
            text = getattr(part, "text", None)
            if not text:
                continue
            is_thought = False
            if hasattr(part, "thought"):
                is_thought = bool(getattr(part, "thought"))
            else:
                role = getattr(part, "role", "")
                is_thought = (role == "thought")

            if is_thought:
                thoughts.append(text.strip())
            else:
                answers.append(text.strip())

    except Exception:
        # 兜底：尝试从顶层 text
        text_fallback = getattr(response, "text", None)
        if text_fallback:
            answers.append(text_fallback.strip())

    answer = "\n".join(a for a in answers if a)
    return thoughts, answer


In [11]:
def parse_json_answer(answer_text: str) -> Dict[str, Any]:
    # 1) 尝试严格 JSON
    try:
        obj = json.loads(answer_text)
        if isinstance(obj, dict) and "is_bug" in obj and "reason" in obj:
            return {"is_bug": bool(obj["is_bug"]), "reason": str(obj["reason"])}
    except Exception:
        pass

    # 2) 宽松提取（非常简化；可按需增强）
    lower = answer_text.lower()
    is_bug = None
    if "is_bug" in lower:
        seg = lower.split("is_bug", 1)[-1][:80]
        if "true" in seg:
            is_bug = True
        elif "false" in seg:
            is_bug = False

    reason = ""
    if "reason" in lower:
        tail = answer_text.split("reason", 1)[-1]
        tail = tail.lstrip(" \t\n\r:：-")
        reason = tail.strip().split("\n", 1)[0][:500]

    result = {}
    if is_bug is not None:
        result["is_bug"] = is_bug
    if reason:
        result["reason"] = reason
    return result


In [12]:
class TransientError(Exception):
    pass

@retry(
    reraise=True,
    stop=stop_after_attempt(4),
    wait=wait_exponential(multiplier=1.0, min=1, max=8),
    retry=retry_if_exception_type(TransientError),
)
def call_gemini_with_thoughts(prompt: str):
    try:
        resp = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt,
            config=GEN_CONFIG
        )
        if not getattr(resp, "candidates", None):
            raise TransientError("Empty candidates in response")
        return resp
    except Exception as e:
        # 未知错误暂按可重试处理
        raise TransientError(str(e))


In [ ]:
def process_file_with_gemini(
    input_path: str,
    output_path: str,
    excluded_ids: Optional[set] = None,
    max_elements: int = MAX_ELEMENTS,
    rate_pause_sec: float = RATE_PAUSE_SEC,
    keep_raw: bool = True,
    fail_sleep_sec: int = FAIL_SLEEP_SEC,
    max_consec_fail: int = MAX_CONSEC_FAIL
) -> None:
    """
    输入：JSON 列表，每个元素至少包含：
        - gen_trace: trace 内容
        - 可能还有 reason / 其它字段（保留）
    输出：保留原字段，并增加 thoughts / answer / parsed / raw
    失败策略：
        - 单次失败后 sleep(fail_sleep_sec)
        - 连续失败 >= max_consec_fail 时提前保存并退出
    """
    excluded_ids = excluded_ids or set()

    with open(input_path, "r", encoding="utf-8") as f:
        all_data = json.load(f)
        if not isinstance(all_data, list):
            raise ValueError("输入文件必须是 JSON 列表")

    results = []
    processed = 0
    consec_fail = 0

    for idx, item in enumerate(tqdm(all_data, desc="Processing traces"), start=1):
        if processed >= max_elements:
            break

        # 自增编号作为 entry_id
        entry_id = idx
        
        if entry_id in excluded_ids:
            continue

        gen_trace = (item.get("gen_trace") or "").strip()
        if not gen_trace:
            continue

        prompt = INSTRUCTION + gen_trace

        try:
            response = call_gemini_with_thoughts(prompt)
            thoughts, answer = extract_thoughts_and_answer(response)
            parsed = parse_json_answer(answer)

            out_obj = {
                **item,              # 保留原字段，包括 gen_trace / reason
                "entry_id": entry_id,
                "thoughts": thoughts,
                "answer": answer,
                "parsed": parsed,
            }

            if keep_raw:
                try:
                    out_obj["raw"] = json.loads(response.to_json())
                except Exception:
                    pass

            results.append(out_obj)
            processed += 1
            consec_fail = 0  # 成功重置连续失败计数
            time.sleep(rate_pause_sec)

        except Exception as e:
            consec_fail += 1
            results.append({
                **item,
                "entry_id": entry_id,
                "error": str(e)
            })
            print(f"[✗] Error on entry={entry_id}: {e} (consecutive fails={consec_fail})")
            time.sleep(fail_sleep_sec)

            if consec_fail >= max_consec_fail:
                print(f"⚠️ 连续失败 {max_consec_fail} 次，提前终止。")
                break

    # 保存当前结果
    with open(output_path, "w", encoding="utf-8") as f_out:
        json.dump(results, f_out, ensure_ascii=False, indent=2)

    print(f"✅ 已处理 {processed} 条记录（共写入 {len(results)} 条）保存至: {output_path}")


In [14]:
INPUT_PATH  = "/Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/dataset_gen_new/gen_1-step/trace_gen_bugfree.json"   # 形如 [{"id": "...", "gen_trace": "...", "reason": "...", ...}, ...]
OUTPUT_PATH = "/Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/dataset_gen_new/gen_1-step/trace_gen_bugfree_thinking_summary_by_gemini.json"

process_file_with_gemini(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    excluded_ids=EXCLUDED_IDS,   # 如需排除：set(["id_1", "id_2"])
    max_elements=MAX_ELEMENTS,
    rate_pause_sec=RATE_PAUSE_SEC,
    keep_raw=True,               # 明确 True
    fail_sleep_sec=FAIL_SLEEP_SEC,
    max_consec_fail=MAX_CONSEC_FAIL
)


Processing traces:   0%|          | 0/1773 [00:00<?, ?it/s]

Processing traces: 100%|██████████| 1773/1773 [7:06:29<00:00, 14.43s/it]  

✅ 已处理 1773 条记录（共写入 1773 条）保存至: /Users/poyu/Documents/ollm_google_drive/NCF_new(after0530)/dataset_gen_new/gen_1-step/trace_gen_bugfree_thinking_summary_by_gemini.json
